[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/029_pytorch_datasets/pytorch_datasets.ipynb)

# Pytorch - Datasets

En los posts anteriores hemos introducido los conceptos fundamentales de la librería de `Deep Learning` `Pytorch` y también hemos visto la funcionalidad que nos ofrece a la hora de diseñar y entrenar `redes neuronales`. En este post nos enfocamos en la herramientas que la librería nos da a la hora definir nuestros *datasets*.

In [1]:
import torch
import numpy as np
from google.colab import drive

# Montamos Google Drive para acceder al dataset
drive.mount('/content/drive')

Mounted at /content/drive


## Iterando tensores

En los posts anteriores hemos utilizado el dataset MNIST para ilustrar los diferentes ejemplos que hemos visto. Vamos a seguir con este caso. A continuación tenemos una implementación en la que iteramos por los datos de manera explícita para entrenar nuestra red.

In [2]:
# ------------------------------------------------------------
# Carga del dataset CIFAR-10 desde Drive
# Ruta proporcionada por el usuario
# ------------------------------------------------------------
import pickle
import os

ruta = '/content/drive/MyDrive/MachineLearning/datasets/mnist6'

def unpickle(file):
    """Función auxiliar para cargar los archivos pickle de CIFAR-10"""
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

# Verificamos qué archivos hay en la carpeta
print("Archivos encontrados en la ruta:")
print(os.listdir(ruta))

# Cargamos los 5 batches de entrenamiento
X_list = []
y_list = []

for i in range(1, 6):
    batch_path = os.path.join(ruta, f'data_batch_{i}')
    if not os.path.exists(batch_path):
        # Por si los archivos no tienen extensión o tienen otro nombre
        batch_path = os.path.join(ruta, f'data_batch_{i}.bin')  # por si acaso
    batch = unpickle(batch_path)

    # CIFAR-10 usa keys en bytes
    data_key = b'data' if b'data' in batch else 'data'
    labels_key = b'labels' if b'labels' in batch else 'labels'

    X_list.append(batch[data_key])          # shape (10000, 3072)
    y_list.append(batch[labels_key])        # lista de 10000 etiquetas

# Concatenamos todo el train
X_train = np.concatenate(X_list, axis=0).astype(np.float32) / 255.0   # normalizamos a [0,1]
y_train = np.array(y_list).flatten().astype(np.int64)

# Cargamos el batch de test (si existe)
test_path = os.path.join(ruta, 'test_batch')
if not os.path.exists(test_path):
    test_path = os.path.join(ruta, 'test_batch.bin')

if os.path.exists(test_path):
    test_batch = unpickle(test_path)
    data_key = b'data' if b'data' in test_batch else 'data'
    labels_key = b'labels' if b'labels' in test_batch else 'labels'
    X_test = test_batch[data_key].astype(np.float32) / 255.0
    y_test = np.array(test_batch[labels_key]).astype(np.int64)
else:
    print("¡Advertencia! No se encontró test_batch. Usaremos una parte de train como test (solo para demostración).")
    # Fallback: usamos los últimos 10000 de train como test
    X_test = X_train[-10000:]
    y_test = y_train[-10000:]
    X_train = X_train[:-10000]
    y_train = y_train[:-10000]

print("\nX_train shape:", X_train.shape)   # debería ser (50000, 3072)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)       # debería ser (10000, 3072)
print("y_test shape:", y_test.shape)


Archivos encontrados en la ruta:
['cifar-10-batches-py', 'batches.meta', 'data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'cifar-10-python.tar.gz']

X_train shape: (50000, 3072)
y_train shape: (50000,)
X_test shape: (10000, 3072)
y_test shape: (10000,)


In [3]:
# Convertimos a tensores de PyTorch y los movemos a GPU (si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

X_t = torch.from_numpy(X_train).float().to(device)
Y_t = torch.from_numpy(y_train).long().to(device)

Usando dispositivo: cuda


In [4]:
from sklearn.metrics import accuracy_score

def softmax(x):
    # Softmax estable para convertir logits en probabilidades
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

def evaluate(x):
    # Evaluación del modelo en modo eval (sin gradientes)
    model.eval()
    with torch.no_grad():
        y_pred = model(x)
        y_probas = softmax(y_pred)
        return torch.argmax(y_probas, axis=1)

In [5]:
# ------------------------------------------------------------
# Modelo simple: MLP de una capa oculta
# D_in = 3072 porque cada imagen CIFAR-10 es 32x32x3 = 3072
# ------------------------------------------------------------
D_in, H, D_out = 3072, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)   # lr un poco más bajo que en MNIST

epochs = 20          # menos epochs porque el dataset es más complejo
log_each = 5
l = []
model.train()

for e in range(1, epochs+1):
    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

# Evaluación en test
y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
print("Accuracy test:", accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 5/20 Loss 2.28409
Epoch 10/20 Loss 2.26121
Epoch 15/20 Loss 2.24069
Epoch 20/20 Loss 2.22623
Accuracy test: 0.2417


## Iterando por *Batches*

En la implementación anterior estamos optimizando nuestro modelo con el algoritmo de `batch gradient descent`, en el que utilizamos todos nuestros datos en cada paso de optimización. Sin embargo, un algoritmo que puede converger más rápido (y única opción si nuestro dataset es tan grande que no cabe en memoria) es el de `mini-batch gradient descent` (el cual hemos ya utilizado en posts anteriores).

In [6]:
# ------------------------------------------------------------
# Entrenamiento con mini-batches (implementación manual)
# ------------------------------------------------------------
D_in, H, D_out = 3072, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 10
batch_size = 128
log_each = 1
l = []
model.train()
batches = len(X_t) // batch_size

for e in range(1, epochs+1):
    _l = []
    # iteramos por batches
    for b in range(batches):
        x_b = X_t[b*batch_size:(b+1)*batch_size]
        y_b = Y_t[b*batch_size:(b+1)*batch_size]

        # forward
        y_pred = model(x_b)

        # loss
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
print("Accuracy test:", accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 1/10 Loss 1.98390
Epoch 2/10 Loss 1.89771
Epoch 3/10 Loss 1.84262
Epoch 4/10 Loss 1.80277
Epoch 5/10 Loss 1.77130
Epoch 6/10 Loss 1.74565
Epoch 7/10 Loss 1.72350
Epoch 8/10 Loss 1.70451
Epoch 9/10 Loss 1.68765
Epoch 10/10 Loss 1.67249
Accuracy test: 0.436


Si bien esta implementación es correcta y funcional, dependiendo de nuestros datos puede llegar a complicarse mucho (por ejemplo, si necesitamos cargar muchas imágenes a las cuales queremos aplicar transformaciones, juntar en batches, etc...). Además, es común reutilizar la lógica para cargar nuestros datos no sólo para entrenar la red, si no para generar predicciones. Este hecho motiva el uso de las clases especiales que `Pytorch` nos ofrece para ello.

## La clase *Dataset*

La primera clase que tenemos que conocer es la clase `Dataset`. Esta clase hereda de la clase madre `torch.utils.data.Dataset` y tenemos que definir, como mínimo, tres funciones:

- `__init__`: el constructor
- `__len__`: devuelve el número de muestras en el dataset
- `__getitem__`: devuelve una muestra en concreto del dataset

Una vez definida la clase, ésta puede usarse como si de cualquier iterador se tratase.

In [7]:
# ------------------------------------------------------------
# Clase Dataset personalizada
# Hereda de torch.utils.data.Dataset
# ------------------------------------------------------------
class DatasetPersonalizado(torch.utils.data.Dataset):
    # constructor
    def __init__(self, X, Y):
        # Convertimos a tensores y los dejamos en el dispositivo (GPU/CPU)
        self.X = torch.from_numpy(X).float().to(device)
        self.Y = torch.from_numpy(Y).long().to(device)

    # devolvemos el número de datos en el dataset
    def __len__(self):
        return len(self.X)

    # devolvemos el elemento `ix` del dataset
    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

Una vez definida la clase, podemos instanciar un objeto que podemos usar para iterar por nuestros datos.

In [8]:
dataset = DatasetPersonalizado(X_train, y_train)
print("Tamaño del dataset:", len(dataset))

Tamaño del dataset: 50000


In [9]:
# ------------------------------------------------------------
# Entrenamiento usando el Dataset (acceso por slicing)
# ------------------------------------------------------------
D_in, H, D_out = 3072, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 10
batch_size = 128
log_each = 1
l = []
model.train()
batches = len(dataset) // batch_size

for e in range(1, epochs+1):
    _l = []
    for b in range(batches):
        # Accedemos al batch usando slicing sobre el Dataset
        x_b, y_b = dataset[b*batch_size:(b+1)*batch_size]

        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
print("Accuracy test:", accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 1/10 Loss 1.98927
Epoch 2/10 Loss 1.90167
Epoch 3/10 Loss 1.85005
Epoch 4/10 Loss 1.81193
Epoch 5/10 Loss 1.78171
Epoch 6/10 Loss 1.75664
Epoch 7/10 Loss 1.73513
Epoch 8/10 Loss 1.71631
Epoch 9/10 Loss 1.69986
Epoch 10/10 Loss 1.68505
Accuracy test: 0.4372


Podemos iterar directamente sobre el objeto `dataset` de la misma manera que hacíamos anteriormente, sin embargo `Pytorch` no ofrece otro objeto que nos facilita las cosas a la hora de iterar por batches.

## La clase *DataLoader*

La clase `DataLoader` recibe un `Dataset` e implementa la lógica para iterar nuestros datos en batches.

In [10]:
# Creamos el DataLoader. shuffle=True mezcla los datos al inicio de cada epoch
dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

In [11]:
# Obtenemos un batch de ejemplo
x, y = next(iter(dataloader))
print("Shape del batch de imágenes:", x.shape)   # (batch_size, 3072)
print("Shape del batch de etiquetas:", y.shape)  # (batch_size,)

Shape del batch de imágenes: torch.Size([128, 3072])
Shape del batch de etiquetas: torch.Size([128])


También permite mezclar los datos al principio de cada epoch con el parámetro `shuffle`, de manera automática carga nuestros datos de manera optimizada utilizando varios *cores* de nuestra CPU si es posible, etc.

In [12]:
# ------------------------------------------------------------
# Entrenamiento usando DataLoader (forma recomendada)
# ------------------------------------------------------------
D_in, H, D_out = 3072, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 10
log_each = 1
l = []
model.train()

for e in range(1, epochs+1):
    _l = []
    # iteramos por batches en el dataloader
    for x_b, y_b in dataloader:
        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
print("Accuracy test:", accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 1/10 Loss 1.98943
Epoch 2/10 Loss 1.89948
Epoch 3/10 Loss 1.84529
Epoch 4/10 Loss 1.80602
Epoch 5/10 Loss 1.77592
Epoch 6/10 Loss 1.75095
Epoch 7/10 Loss 1.72991
Epoch 8/10 Loss 1.71188
Epoch 9/10 Loss 1.69560
Epoch 10/10 Loss 1.68098
Accuracy test: 0.4364


También permite definir nuestra propia lógica para crear los batches, algo que puede ser útil en ciertas ocasiones.

In [13]:
def collate_fn(batch):
    # Función personalizada para crear el batch.
    # Recibe una lista de tuplas (x, y) y devuelve tensores apilados.
    return torch.stack([x for x, y in batch]), torch.stack([y for x, y in batch])

In [14]:
# DataLoader con collate_fn personalizado
dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)

In [15]:
# ------------------------------------------------------------
# Entrenamiento final con DataLoader + collate_fn
# ------------------------------------------------------------
D_in, H, D_out = 3072, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 10
log_each = 1
l = []
model.train()

for e in range(1, epochs+1):
    _l = []
    for x_b, y_b in dataloader:
        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

        # Guardamos un checkpoint al final de cada epoch (opcional)
        PATH = f"./checkpoint_{e}.pt"
        torch.save(model.state_dict(), PATH)

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
print("Accuracy test:", accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 1/10 Loss 1.98020
Epoch 2/10 Loss 1.88987
Epoch 3/10 Loss 1.83560
Epoch 4/10 Loss 1.79659
Epoch 5/10 Loss 1.76499
Epoch 6/10 Loss 1.73894
Epoch 7/10 Loss 1.71730
Epoch 8/10 Loss 1.69867
Epoch 9/10 Loss 1.68205
Epoch 10/10 Loss 1.66689
Accuracy test: 0.4369


In [16]:
PATH = './checkpoint_final.pt'
torch.save(model.state_dict(), PATH)
print("Modelo guardado en", PATH)

Modelo guardado en ./checkpoint_final.pt


In [ ]:
# guardar modelo

PATH = './checkpoint.pt'
torch.save(model.state_dict(), PATH)

## Resumen

En este post hemos visto diferentes maneras en las que podemos iterar por nuestros datos para entrenar un modelo en `Pytorch`. Si nuestro dataset es sencillo y podemos representarlo como un simple `array` de `NumPy` podemos iterar directamente el `array`, transformándolo previamente en un `tensor`. Sin embargo, cuando nuestro dataset sea más grande y no quepa en memoria o necesite cierto pre-proceso o transformaciones, es muy conveniente utilizar las clases que `Pytorch` nos ofrece para ello. Estas clases son, principalmente, el `Dataset` y el `DataLoader`, las cuales nos van a permitir iterar por nuestros datos de manera eficiente y generar *batches* de forma sencilla (además de otras funcionalidades como mezclar los datos al principio de cada epoch, cargar datos en paralelo, etc).